In [ ]:
import pandas as pd
from typing import cast
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from data_preprocessing import create_train_test_val_sets, read_processed_data
from copy import deepcopy
import joblib


In [ ]:
#Create test train splits
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil= read_processed_data(r"..\data\processed\phiusiil_processed.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
phiusiil_sets = create_train_test_val_sets(x_phiusiil,y_phiusiil, label_col="Label", test_size=0.2, n_splits=5)

### Tuning Classifiers

In [ ]:
#XGBoost

def optimize_xgboost(X: pd.DataFrame, y: pd.Series, dataset: str, splits) -> XGBClassifier:
    """
    Returns a trained and tuned XGBClassifier
    
    Parameters:
    X: input data
    y: target variable
    dataset: which dataset is being used to tune the XGBClassifier
    splits: A list of tuples containing the splits for CV

    Returns:
    Tuned XGBClassifier
    """
    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    if dataset == 'mendeley':
        scale_weights.append(counts[1]/counts[0])
    elif dataset == 'phiusiil':
        scale_weights.append(counts[0]/counts[1])
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'gamma': [0.1, 0.2],
        'subsample': [0.6, 0.7],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [300, 500, 750, 1000, 1250],
        'scale_pos_weight': scale_weights
    }

    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=splits)
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return cast(XGBClassifier, random_search.best_estimator_)

print("Running hyperparameter tuning using Mendeley Dataset:")
# xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train_val"], mendeley_sets["y_train_val"], 'mendeley', mendeley_sets["cv_splits"])
# joblib.dump(xgboost_mendeley, "./models/phase_1/xgboost_mendeley_no_fs.joblib")
xgboost_mendeley = XGBClassifier(subsample=0.7, scale_pos_weight=0.93, n_estimators=1000, min_child_weight=3, max_depth=10, learning_rate=0.4, gamma=0.2, colsample_bytree=0.8)

print("Running hyperparameter tuning using Phiusiil Dataset:")
# xgboost_phiusiil = optimize_xgboost(phiusiil_sets["x_train_val"], phiusiil_sets["y_train_val"], 'phiusiil', phiusiil_sets["cv_splits"])
xgboost_phiusiil = XGBClassifier(subsample=0.9, n_estimators=500, min_child_weight=3, max_depth=5, learning_rate=0.01, gamma=0, colsample_bytree=0.9)
# joblib.dump(xgboost_mendeley, "./models/phase_1/xgboost_phiusiil_no_fs.joblib")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
def train_no_feature_selection(dataset, model):
    """
    Predict using the testset and print the confusion matrix and classification report

    Parameters:
    x_test: the test set for input variables
    y_test: the test set for the target variable
    model: the trained model
    """
    model.fit(dataset["x_train_val"], dataset["y_train_val"])
    y_test_pred = model.predict(dataset["x_test"])
    print("\nTest Set Performance")
    cm = confusion_matrix(dataset["y_test"], y_test_pred)
    ConfusionMatrixDisplay(cm).plot()
    print(classification_report(dataset["y_test"], y_test_pred))
    

#create deepcopies of models so they can be reused
xgboost_mendeley_baseline_no_fs = deepcopy(xgboost_mendeley)
xgboost_phiusiil_baseline_no_fs = deepcopy(xgboost_phiusiil)

print('XGBoost Mendeley Results:')
train_no_feature_selection(mendeley_sets, xgboost_mendeley_baseline_no_fs)

print('XGBoost Phiusiil Results:')
train_no_feature_selection(phiusiil_sets, xgboost_phiusiil_baseline_no_fs)

In [ ]:
import pandas as pd
import numpy as np
from copy import deepcopy

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression



def sanitize_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Clean processed feature matrix loaded from CSV.
    """
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    return X



def optimize_logistic_regression(X, y, splits):
    """
    Tune Logistic Regression using cross-validation splits.

    Parameters:
    X: input features
    y: target labels
    splits: list of (train_idx, val_idx) tuples for CV

    Returns:
    best fitted pipeline
    """
    params = {
        "model__C": [0.01, 0.1, 1, 10]
    }

    pipeline = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        ))
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=splits,
        scoring="f1",
        n_jobs=1
    )

    grid_search.fit(X, y)

    print("\nBest hyperparameters:")
    print(grid_search.best_params_)
    print("Best CV F1:", grid_search.best_score_)

    return grid_search.best_estimator_


# Clean NaNs from processed sparse CSVs
x_mendeley = sanitize_features(x_mendeley)
x_phiusiil = sanitize_features(x_phiusiil)

# # --------------------------------------------------
# # Create train/test splits
# # --------------------------------------------------
mendeley_sets = create_train_test_val_sets(
    x_mendeley, y_mendeley, label_col="Label", test_size=0.2, n_splits=5
)

phiusiil_sets = create_train_test_val_sets(
    x_phiusiil, y_phiusiil, label_col="Label", test_size=0.2, n_splits=5
)

# --------------------------------------------------
# Logistic Regression with tuned value C=10
# --------------------------------------------------
logreg_mendeley = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("model", LogisticRegression(
        C=10,
        solver="saga",
        class_weight="balanced",
        max_iter=5000,
        tol=1e-3,
        random_state=42
    ))
])

joblib.dump(logreg_mendeley, "./models/phase_1/logreg_mendeley_no_fs.joblib")

logreg_phiusiil = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("model", LogisticRegression(
        C=10,
        solver="saga",
        class_weight="balanced",
        max_iter=5000,
        tol=1e-3,
        random_state=42
    ))
])

joblib.dump(logreg_phiusiil, "./models/phase_1/logreg_phiusiil_no_fs.joblib")

# --------------------------------------------------
# Reuse existing evaluation function
# train_no_feature_selection(dataset, model)
# --------------------------------------------------

# print("Running hyperparameter tuning using Mendeley Dataset:")
# logreg_mendeley = optimize_logistic_regression(
#     mendeley_sets["x_train_val"],
#     mendeley_sets["y_train_val"],
#     mendeley_sets["cv_splits"]
# )

# print("Running hyperparameter tuning using Phiusiil Dataset:")
# logreg_phiusiil = optimize_logistic_regression(
#     phiusiil_sets["x_train_val"],
#     phiusiil_sets["y_train_val"],
#     phiusiil_sets["cv_splits"]
# )


print("Logistic Regression Mendeley Results:")
train_no_feature_selection(mendeley_sets, logreg_mendeley)

print("Logistic Regression Phiusiil Results:")
train_no_feature_selection(phiusiil_sets, logreg_phiusiil)